# Data Pilot 

## Processo ETL

1. **Inspecionar a fonte:** conhecer estrutura, qualidade, distribuição e diferenças entre grupos antes de transformar os dados.
2. **Extract (extração):** ler o arquivo CSV original para a área de trabalho do notebook.
3. **Transform (transformação):** padronizar textos e tipos, tratar valores ausentes/inválidos e criar colunas derivadas.
4. **Load (carga):** salvar a tabela preparada em `data_processed/sales_clean.csv`, pronta para análises e consumo pelo painel.

A inspeção inicial abaixo é um diagnóstico da fonte e antecede a extração formal do fluxo ETL.

### 1. Inspeção exploratória da fonte (antes do Extract)

A análise rápida ajuda a entender o que existe antes de alterar qualquer registro. Usaremos contagens, estatísticas descritivas e comparações de grupos. Nesta etapa, as contagens de status ainda podem aparecer separadas por diferenças de capitalização.

In [2]:
from pathlib import Path
import pandas as pd

ARQUIVO_ORIGEM = Path("data/sales.csv")

# Leitura apenas para diagnóstico inicial da fonte; a extração do ETL será identificada adiante.
raw_preview = pd.read_csv(ARQUIVO_ORIGEM)
print(f"Dimensão: {raw_preview.shape[0]} linhas × {raw_preview.shape[1]} colunas")
display(raw_preview.head())
display(raw_preview.dtypes.rename("tipo inferido pelo pandas").to_frame())
display(raw_preview.isna().sum().rename("valores ausentes").to_frame())

Dimensão: 500 linhas × 10 colunas


,pedido_id,data,produto,categoria,cliente_id,cidade,canal,quantidade,preco_unitario,status
0,PED-0001,2025-07-05,Mouse Precision,Acessórios,CLI-019,Manaus,marketplace,1,369.70,concluido
1,PED-0002,2025-07-25,Notebook Pulse 14,Eletrônicos,CLI-049,Recife,aplicativo,2,1381.95,CONCLUIDO
2,PED-0003,2025-05-30,Garrafa Térmica,Utilidades,CLI-071,Manaus,loja física,1,1483.04,cancelado
3,PED-0004,2025-11-29,Suporte para Notebook,Utilidades,CLI-036,São Paulo,aplicativo,5,527.13,Concluido
4,PED-0005,2025-03-24,Notebook Pulse 14,Eletrônicos,CLI-041,Brasília,aplicativo,4,1494.22,concluido


,tipo inferido pelo pandas
pedido_id,str
data,str
produto,str
categoria,str
cliente_id,str
cidade,str
canal,str
quantidade,int64
preco_unitario,float64
status,str


,valores ausentes
pedido_id,0
data,0
produto,0
categoria,0
cliente_id,0
cidade,0
canal,0
quantidade,0
preco_unitario,0
status,0


#### Tipos de variáveis e distribuição

- **Qualitativas nominais:** `pedido_id`, `produto`, `categoria`, `cliente_id`, `cidade`, `canal` e `status` identificam ou classificam registros sem ordem natural. IDs são rótulos, mesmo quando incluem números.
- **Data temporal:** `data` registra quando ocorreu o pedido; será convertida para `datetime`.
- **Quantitativa discreta:** `quantidade` é uma contagem de unidades.
- **Quantitativa contínua:** `preco_unitario` e a futura `receita` são valores numéricos mensuráveis.

`describe` resume as variáveis numéricas; as frequências mostram a distribuição das categorias.

In [3]:
display(raw_preview.describe())
for coluna in ["status", "categoria", "canal"]:
    print(f"\nDistribuição de {coluna} (fonte original):")
    display(raw_preview[coluna].value_counts(dropna=False).rename_axis(coluna).to_frame("registros"))

,quantidade,preco_unitario
count,500.000000,500.00000
mean,3.102000,731.96102
std,1.444912,436.14497
min,1.000000,13.49000
25%,2.000000,356.36250
50%,3.000000,734.94000
75%,4.000000,1112.01750
max,5.000000,1494.22000



Distribuição de status (fonte original):


,registros
status,
CONCLUIDO,106
cancelado,104
Cancelado,101
concluido,96
Concluido,93



Distribuição de categoria (fonte original):


,registros
categoria,
Acessórios,135
Utilidades,107
Casa Inteligente,104
Móveis,80
Eletrônicos,74



Distribuição de canal (fonte original):


,registros
canal,
marketplace,136
loja física,132
site,119
aplicativo,113


#### Comparação entre grupos

A tabela abaixo compara quantidade de registros e médias de quantidade/preço por canal de venda. É uma leitura descritiva da fonte, antes da padronização; médias não representam receita e não indicam causalidade.

In [4]:
comparacao_canais = (
    raw_preview.groupby("canal", dropna=False)
    .agg(registros=("pedido_id", "size"),
         quantidade_media=("quantidade", "mean"),
         preco_medio=("preco_unitario", "mean"))
    .sort_values("registros", ascending=False)
)
display(comparacao_canais.round(2))

,registros,quantidade_media,preco_medio
canal,,,
marketplace,136,3.18,751.38
loja física,132,3.03,697.90
site,119,3.16,696.22
aplicativo,113,3.04,786.02


### 2. Extract

Lemos a fonte original para um DataFrame de trabalho. Mantemos uma cópia da leitura bruta para comparação e não sobrescrevemos o CSV de origem.

In [12]:
df = pd.read_csv(ARQUIVO_ORIGEM)
print(f"Extração concluída: {len(df)} registros e {len(df.columns)} colunas")

Extração concluída: 500 registros e 10 colunas


### 3. Transform


#### 3.1 Diagnóstico de qualidade

Antes de limpar, verificamos ausências e duplicatas. O arquivo atual pode não conter nulos; as regras abaixo continuam úteis quando chegarem novos arquivos.

In [6]:
print("Ausentes por coluna:")
display(df.isna().sum().rename("ausentes").to_frame())
print(f"Linhas totalmente duplicadas: {df.duplicated().sum()}")
print("Tipos inferidos:")
display(df.dtypes.rename("tipo").to_frame())

Ausentes por coluna:


,ausentes
pedido_id,0
data,0
produto,0
categoria,0
cliente_id,0
cidade,0
canal,0
quantidade,0
preco_unitario,0
status,0


Linhas totalmente duplicadas: 0
Tipos inferidos:


,tipo
pedido_id,str
data,str
produto,str
categoria,str
cliente_id,str
cidade,str
canal,str
quantidade,int64
preco_unitario,float64
status,str


#### 3.2 Padronização e tratamento de ausentes

- Removemos duplicatas exatas e espaços externos nos campos textuais.
- Datas inválidas viram `NaT`; registros sem identificador do pedido ou data válida são removidos, pois não podem ser analisados com segurança.
- Ausências em campos descritivos recebem `Não informado`, preservando o registro sem fingir que sabemos a categoria real.
- Quantidade e preço são convertidos para números. Valores não numéricos ou não positivos são considerados inválidos e removidos, pois não representam uma venda válida.
- Ausências em identificadores são removidas: não imputamos IDs artificiais.

Nenhuma linha é removida por nulo no arquivo atual, mas o tratamento fica preparado para dados futuros.

In [7]:
df = df.drop_duplicates().copy()

colunas_texto = ["pedido_id", "produto", "categoria", "cliente_id", "cidade", "canal", "status"]
for coluna in colunas_texto:
    df[coluna] = df[coluna].astype("string").str.strip()
    df[coluna] = df[coluna].replace("", pd.NA)

# Tipos apropriados antes de aplicar as regras de qualidade.
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["quantidade"] = pd.to_numeric(df["quantidade"], errors="coerce")
df["preco_unitario"] = pd.to_numeric(df["preco_unitario"], errors="coerce")

linhas_antes = len(df)
df = df.dropna(subset=["pedido_id", "data", "quantidade", "preco_unitario"])
df = df[(df["quantidade"] > 0) & (df["preco_unitario"] > 0)].copy()
linhas_removidas = linhas_antes - len(df)

colunas_descritivas = ["produto", "categoria", "cliente_id", "cidade", "canal", "status"]
df[colunas_descritivas] = df[colunas_descritivas].fillna("Não informado")

print(f"Registros removidos por campos essenciais ausentes/inválidos ou valores não positivos: {linhas_removidas}")
print("Ausentes restantes:")
display(df.isna().sum().rename("ausentes").to_frame())

Registros removidos por campos essenciais ausentes/inválidos ou valores não positivos: 0
Ausentes restantes:


,ausentes
pedido_id,0
data,0
produto,0
categoria,0
cliente_id,0
cidade,0
canal,0
quantidade,0
preco_unitario,0
status,0


#### 3.3 Padronização de categorias e criação de variáveis

Padronizamos os rótulos de status para facilitar filtros. `mes` é uma variável temporal agregada, útil em séries mensais. `receita` é uma variável quantitativa contínua derivada de quantidade × preço unitário. A receita bruta por linha é mantida para auditoria; as funções de negócio filtram pedidos concluídos.

In [8]:
df["status"] = df["status"].str.casefold().replace({
    "concluído": "concluido",
    "concluido": "concluido",
    "cancelado": "cancelado",
})

# Mantém rótulos desconhecidos visíveis; não os classifica silenciosamente como concluídos/cancelados.
df["mes"] = df["data"].dt.to_period("M").astype("string")
df["receita"] = df["quantidade"] * df["preco_unitario"]

print("Status padronizados:")
display(df["status"].value_counts(dropna=False).to_frame("registros"))
print("Tipos após transformação:")
display(df.dtypes.rename("tipo").to_frame())

Status padronizados:


,registros
status,
concluido,295
cancelado,205


Tipos após transformação:


,tipo
pedido_id,string
data,datetime64[us]
produto,string
categoria,string
cliente_id,string
cidade,string
canal,string
quantidade,int64
preco_unitario,float64
status,string


### 4. Load

Salvamos o conjunto transformado em CSV separado por ponto e vírgula, com vírgula decimal. Essa tabela pode ser carregada depois pelo painel. A pasta de saída é criada se ainda não existir.

In [9]:
PASTA_SAIDA = Path("data_processed")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVO_TRATADO = PASTA_SAIDA / "sales_clean.csv"
df.to_csv(ARQUIVO_TRATADO, index=False, sep=";", decimal=",", date_format="%Y-%m-%d")
print(f"Carga concluída: {ARQUIVO_TRATADO} ({len(df)} registros)")

Carga concluída: data_processed\sales_clean.csv (500 registros)


### 5. Métricas de vendas

As funções abaixo isolam KPIS do negócio. Receita, pedidos, produtos vendidos, clientes e ticket médio consideram apenas pedidos concluídos. A taxa de cancelamento usa pedidos distintos no denominador para não contar duas vezes um mesmo pedido.

In [10]:
def pedidos_concluidos(dados: pd.DataFrame) -> pd.DataFrame:
    """Retorna as linhas de pedidos concluídos."""
    return dados.loc[dados["status"] == "concluido"].copy()


def receita_total(dados: pd.DataFrame) -> float:
    """Soma a receita das linhas de pedidos concluídos."""
    return float(pedidos_concluidos(dados)["receita"].sum())


def total_pedidos(dados: pd.DataFrame) -> int:
    """Conta pedidos concluídos distintos."""
    return int(pedidos_concluidos(dados)["pedido_id"].nunique())


def produtos_vendidos(dados: pd.DataFrame) -> int:
    """Soma unidades vendidas em pedidos concluídos."""
    return int(pedidos_concluidos(dados)["quantidade"].sum())


def clientes_unicos(dados: pd.DataFrame) -> int:
    """Conta clientes distintos com pedidos concluídos."""
    return int(pedidos_concluidos(dados)["cliente_id"].nunique())


def ticket_medio(dados: pd.DataFrame) -> float:
    """Receita concluída dividida pela quantidade de pedidos concluídos."""
    pedidos = total_pedidos(dados)
    return receita_total(dados) / pedidos if pedidos else 0.0


def taxa_cancelamento(dados: pd.DataFrame) -> float:
    """Percentual de pedidos cancelados entre todos os pedidos distintos."""
    total = dados["pedido_id"].nunique()
    cancelados = dados.loc[dados["status"] == "cancelado", "pedido_id"].nunique()
    return (cancelados / total * 100) if total else 0.0


def receita_por_mes(dados: pd.DataFrame) -> pd.DataFrame:
    """Agrega receita concluída por mês, em ordem cronológica."""
    return (pedidos_concluidos(dados).groupby("mes", as_index=False)["receita"]
            .sum().sort_values("mes"))

In [11]:
resumo = {
    "receita_total": receita_total(df),
    "pedidos_concluidos": total_pedidos(df),
    "unidades_vendidas": produtos_vendidos(df),
    "clientes_unicos": clientes_unicos(df),
    "ticket_medio": ticket_medio(df),
    "taxa_cancelamento_pct": taxa_cancelamento(df),
}
display(pd.Series(resumo, name="valor").to_frame())
display(receita_por_mes(df))

,valor
receita_total,653416.960000
pedidos_concluidos,295.000000
unidades_vendidas,890.000000
clientes_unicos,94.000000
ticket_medio,2214.972746
taxa_cancelamento_pct,41.000000


,mes,receita
0,2025-01,44506.19
1,2025-02,46705.60
2,2025-03,41956.85
3,2025-04,57568.92
4,2025-05,60088.68
5,2025-06,47450.12
6,2025-07,62217.73
7,2025-08,47156.26
8,2025-09,64338.78
9,2025-10,56678.77


## 6. Como o código do projeto se organiza

O notebook documenta a análise e o ETL. A aplicação final está separada em arquivos dentro da pasta src/, para que cada parte tenha uma responsabilidade clara.

### Preparação dos dados — src/preprocess.py

Este módulo lê data/sales.csv, remove duplicatas, converte tipos, aplica regras para valores ausentes/inválidos, padroniza status, cria mes e receita e salva data_processed/sales_clean.csv. É o fluxo Extract → Transform → Load que prepara a fonte para o painel.

Trecho ilustrativo de src/:

    # Bloco representativo do ETL em src/preprocess.py
    # Extração
    ARQUIVO_ORIGEM = Path("data/sales.csv")
    df = pd.read_csv(ARQUIVO_ORIGEM)

    # Transformação: limpeza, padronização e variáveis derivadas
    # Regras detalhadas aplicadas no módulo preprocess.py

    # Carga
    ARQUIVO_TRATADO = Path("data_processed/sales_clean.csv")
    df.to_csv(ARQUIVO_TRATADO, index=False, sep=";", decimal=",")


### Métricas — src/metrics.py

As funções de negócio recebem um DataFrame e retornam indicadores calculados. O painel e o assistente reutilizam essas funções, mantendo uma única definição para cada métrica.

Trecho ilustrativo de src/:

    # Exemplos das funções disponíveis em src/metrics.py
    receita_total(df)
    total_pedidos(df)
    produtos_vendidos(df)
    clientes_unicos(df)
    ticket_medio(df)
    taxa_cancelamento(df)
    receita_por_mes(df)


### Integração de IA — src/llm.py

O provedor usado é o OpenRouter. O modelo é escolhido pela variável OPENROUTER_MODEL; se ela não existir, usa-se o valor definido em DEFAULT_MODEL. A chave de acesso fica em OPENROUTER_API_KEY, carregada do arquivo .env.

Na experiência de chat conectada ao painel, ask_openrouter monta um contexto com métricas e o envia junto da pergunta. run_agent contém também o fluxo de chamadas de ferramentas, no qual o modelo pode solicitar a execução de métricas do código.

Trecho ilustrativo de src/:

    # Escolha do modelo em src/llm.py
    DEFAULT_MODEL = "~openai/gpt-mini-latest"
    model = os.getenv("OPENROUTER_MODEL", DEFAULT_MODEL)

    # O cliente conecta ao provedor OpenRouter
    with OpenRouter(api_key=api_key) as client:
        ...


### Interface — src/app.py

A interface foi construída com Streamlit: filtros de categoria, cartões de indicadores e área de conversa. O gráfico de receita mensal usa Plotly Express e é exibido no Streamlit com st.plotly_chart. O tema visual fica em .streamlit/config.toml.

Trecho ilustrativo de src/:

    # Blocos centrais da interface em src/app.py
    st.pills("Categorias", sorted(df["categoria"].unique()), selection_mode="multi")
    st.metric("Receita total", receita_total(filtered_df))
    fig = px.line(data, x="mes", y="receita", markers=True)
    st.plotly_chart(fig, width="stretch")

    # O chat chama o assistente usando os dados carregados
    answer = ask_openrouter(question, load_data())


### Execução

main.py chama a função principal da aplicação. Também é possível iniciar diretamente a interface com streamlit run src/app.py. O arquivo README.md registra os passos de instalação, execução e configuração.